# Diffusers：扩散模型

Diffusers是视觉生成模型的标准库，提供了丰富的扩散模型实现。

假设你正在做一个AI绘画项目：需要文生图、图生图、图像修复、视频生成等多种能力。如果从论文代码出发，每种模型的实现风格、依赖库、接口设计都不一样，集成起来非常痛苦。Diffusers用统一的Pipeline抽象解决了这个问题——切换模型就像换一个名字，而底层的VAE、UNet、调度器等组件还可以灵活替换。

## Pipeline使用

In [ ]:
from diffusers import StableDiffusionPipeline, DiffusionPipeline
import torch

: 

In [ ]:
# 加载模型
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
)
pipe = pipe.to("cuda")

# 文生图
image = pipe(
    "A photo of a cat wearing sunglasses",
    num_inference_steps=50,
    guidance_scale=7.5,
).images[0]

image.save("output.png")

## 常用Pipeline

Pipeline	功能

StableDiffusionPipeline	文生图
StableDiffusionImg2ImgPipeline	图生图
StableDiffusionInpaintPipeline	图像修复
StableDiffusionXLPipeline	SDXL文生图
StableVideoDiffusionPipeline	图生视频
FluxPipeline	Flux模型

## 核心组件

Diffusers将扩散模型分解为可复用的组件：

In [ ]:
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

In [ ]:
# 各组件独立加载
vae = AutoencoderKL.from_pretrained("model_path", subfolder="vae")
unet = UNet2DConditionModel.from_pretrained("model_path", subfolder="unet")
text_encoder = CLIPTextModel.from_pretrained("model_path", subfolder="text_encoder")
tokenizer = CLIPTokenizer.from_pretrained("model_path", subfolder="tokenizer")
scheduler = DDPMScheduler.from_pretrained("model_path", subfolder="scheduler")

**调度器（Scheduler）** 控制扩散过程：

In [ ]:
from diffusers import DDPMScheduler, DDIMScheduler, EulerDiscreteScheduler

In [ ]:
# 更换调度器
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)

## ControlNet
ControlNet提供精细的图像控制：

In [ ]:
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel

In [ ]:
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_canny",
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16,
)

# 使用Canny边缘图控制
import cv2
canny_image = cv2.Canny(image, 100, 200)

result = pipe(
    "A beautiful landscape",
    image=canny_image,
    num_inference_steps=30,
).images[0]

## LoRA加载

In [ ]:
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
)

# 加载LoRA权重
pipe.load_lora_weights("path/to/lora")

# 调整LoRA强度
pipe.fuse_lora(lora_scale=0.8)

# 多个LoRA叠加
pipe.load_lora_weights("lora1", adapter_name="style")
pipe.load_lora_weights("lora2", adapter_name="character")
pipe.set_adapters(["style", "character"], adapter_weights=[0.5, 0.5])

## 模型训练

Diffusers提供了训练脚本示例：

In [ ]:
from diffusers import DDPMScheduler
from diffusers.optimization import get_cosine_schedule_with_warmup

In [ ]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# 添加噪声
noise = torch.randn_like(latents)
timesteps = torch.randint(0, 1000, (batch_size,))
noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

# 预测噪声
noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample

# 损失计算
loss = F.mse_loss(noise_pred, noise)

## 生态协同

这些库可以无缝协作：

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from accelerate import Accelerator

In [ ]:
# Transformers加载模型
model = AutoModelForCausalLM.from_pretrained("model_name")
tokenizer = AutoTokenizer.from_pretrained("model_name")

# PEFT添加LoRA
lora_config = LoraConfig(r=16, lora_alpha=32, ...)
model = get_peft_model(model, lora_config)

# Accelerate处理分布式
accelerator = Accelerator()
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

# 训练
for batch in dataloader:
    loss = model(**batch).loss
    accelerator.backward(loss)
    optimizer.step()

Hugging Face生态通过模块化设计与统一接口，大幅降低了大模型开发的门槛。Transformers提供模型抽象，PEFT实现高效微调，Accelerate简化分布式训练，Diffusers支持视觉生成——这套工具链已经成为大模型研发的事实标准。回到实际工作流中看：一个典型的微调项目，往往是Transformers加载模型、PEFT添加LoRA适配器、Accelerate处理多卡训练，三者各司其职又无缝衔接。掌握这套生态的用法和边界，能让你在面对具体需求时快速找到最合适的工具组合。